## Day 3: Data Cleaning — Missing Values, Duplicates, Type Conversion & String Operations

Welcome to Day 3!

Yesterday you learned selection, filtering, indexing, and sorting.
Today you learn **data cleaning** — one of the most important skills in data science.

**Think:**

Real-world data is never perfect:
- Some values are missing
- Some rows are duplicates
- Some columns have the wrong type
- Some text has extra spaces or wrong casing

A common estimate is that **80% of a data scientist's time** is spent cleaning data,
not building models. Mastering these tools is essential.

### Day 3 Learning Objectives

By the end of today, you should understand:
- What NaN (missing data) means
- How to detect and count missing values
- How to fill or remove missing values
- How to detect and remove duplicates
- How to replace incorrect values
- How to convert data types
- How to rename columns
- How to clean text columns with `.str` methods
- How to handle outliers with `.clip()`
- A complete data cleaning workflow

---
## Why Data Cleaning Matters

Imagine collecting student records:

| Name | Age | Python |
|------|-----|--------|
| Alice | 20 | 90 |
| Bob | NaN | 85 |
| Charlie | 19 | NaN |
| Alice | 20 | 90 |

**Problems:**
- Missing age for Bob
- Missing Python score for Charlie
- Duplicate Alice row

If we tried to train a Machine Learning model on this, it would fail.
Data cleaning fixes these problems before analysis.

---
## 1. Setup & Dataset

First, import Pandas and NumPy, and create our demonstration dataset.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Pandas version: 2.3.3
NumPy version: 2.4.4


In [ ]:
# Our dataset for Day 3 — contains missing values and duplicates
students = {
    "Name": ["Alice", "Bob", "Charlie", "David", "Alice"],
    "Age": [20, np.nan, 19, 22, 20],
    "Python": [90, 85, np.nan, 70, 90],
    "AI": [88, 75, 92, np.nan, 88],
    "Department": ["CS", "IT", "CS", "SE", "CS"]
}

df = pd.DataFrame(students)
print(df)

---
## PART 1 — What is NaN?

**NaN** stands for **Not a Number**.

It represents missing or undefined data in Pandas/NumPy.

Key points:
- NaN is treated as a `float` value internally
- Any calculation involving NaN returns NaN
- Pandas provides special methods to handle NaN
- `None` in Python is automatically converted to NaN in Pandas

In [ ]:
# Demonstrating NaN behavior
print("NaN is a float:", type(np.nan))
print("NaN + 5 =", np.nan + 5)
print("NaN == NaN?", np.nan == np.nan)  # Surprise: NaN != NaN
print("pd.isna(NaN):", pd.isna(np.nan))

---
## PART 2 — Detect Missing Values

### 2.1 isna()

**Signature:** `DataFrame.isna()`

Returns a DataFrame of the same shape with `True` where values are missing.

In [ ]:
# Detect missing values — each cell becomes True (missing) or False (present)
missing_mask = df.isna()
print("isna() output:")
print(missing_mask)

In [ ]:
# We can also check a single column
print("Age column missing values:")
print(df["Age"].isna())

### 2.2 notna() — Opposite of isna()

Returns `True` where values are **NOT** missing.

In [ ]:
# notna() — True means value is PRESENT
print("notna() output:")
print(df.notna())
print(df["Age"].notna())

### 2.3 isnull() — An Alias

`isnull()` is **exactly the same** as `isna()`.

There is no difference. Pandas provides both names for convenience.

In [ ]:
# isnull() — identical to isna()
print(df.isnull())
print()
print("Are they the same?", df.isna().equals(df.isnull()))
print(df["Python"].isnull())

---
## PART 3 — Count Missing Values

### 3.1 Count per Column

**Signature:** `df.isna().sum()`

Since `True` is treated as 1 and `False` as 0, summing a boolean mask counts the `True` values.

This is **one of the most common Pandas commands** — you will use it constantly.

In [ ]:
# Count missing values per column
missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts)

### 3.2 Total Missing Values

**Signature:** `df.isna().sum().sum()`

Chain two `.sum()` to get the grand total.

In [ ]:
# Total missing values across the entire DataFrame
total_missing = df.isna().sum().sum()
print(f"Total missing values in dataset: {total_missing}")

In [ ]:
# Percentage of missing values per column
missing_percent = (df.isna().sum() / len(df)) * 100
print("Percentage of missing values per column:")
print(missing_percent)

---
## PART 4 — Filling Missing Values

**Signature:** `DataFrame.fillna(value)`

Replaces NaN with a specified value.

### 4.1 Fill with a Constant (e.g., Zero)

In [ ]:
# Fill ALL missing values with 0
df_zero = df.fillna(0)
print("Fill all NaN with 0:")
print(df_zero)

**⚠️ Warning:** Filling everything with 0 is rarely the right choice.
It can distort statistics like mean and standard deviation.

### 4.2 Fill Only One Column

In [ ]:
# Fill missing Age with 21 (default value)
df_copy = df.copy()
df_copy["Age"] = df_copy["Age"].fillna(30)
print("Fill missing Age with 21:")
print(df_copy)

### 4.3 Fill with the Mean

Very common in Machine Learning preprocessing.
The mean preserves the average of the column.

In [ ]:
# Fill missing Age with the mean Age
df_mean = df.copy()
mean_age = df_mean["Age"].mean()
print(f"Mean Age: {mean_age:.2f}")

df_mean["Age"] = df_mean["Age"].fillna(mean_age)
print("\nFill missing Age with mean:")
print(df_mean)

### 4.4 Fill with the Median

Better than mean when the data has **outliers**.
Median is resistant to extreme values.

In [ ]:
# Fill missing Age with the median Age
df_median = df.copy()
median_age = df_median["Age"].median()
print(f"Median Age: {median_age}")

df_median["Age"] = df_median["Age"].fillna(median_age)
print("\nFill missing Age with median:")
print(df_median)

### 4.5 Fill Strings / Categorical Columns

For text/categorical columns, fill with "Unknown" or the most frequent value.

In [ ]:
# Create a version with missing Department
df_dept = df.copy()
df_dept.loc[1, "Department"] = np.nan  # Bob's department becomes NaN

print("Before fill:")
print(df_dept)

# Fill missing Department with "Unknown"
df_dept["Department"] = df_dept["Department"].fillna("Unknown")
print("\nAfter fill with 'Unknown':")
print(df_dept)

In [ ]:
# Fill with the most frequent value (mode)
df_dept2 = df.copy()
df_dept2.loc[1, "Department"] = np.nan

mode_dept = df_dept2["Department"].mode()[0]
print(f"Most frequent department: {mode_dept}")

df_dept2["Department"] = df_dept2["Department"].fillna(mode_dept)
print("\nFill with mode:")
print(df_dept2)

### 4.6 Fill Forward / Backward

Sometimes missing values should take the value from the row before or after.

- `fillna(method="ffill")` — forward fill (use previous value)
- `fillna(method="bfill")` — backward fill (use next value)

In [ ]:
# Demonstrate forward fill
df_ffill = df.copy()
print("Original:")
print(df_ffill)

df_ffill = df_ffill.ffill()
print("\nAfter forward fill:")
print(df_ffill)

In [ ]:
# Demonstrate backward fill
df_bfill = df.copy()
df_bfill = df_bfill.bfill()
print("After backward fill:")
print(df_bfill)

---
## PART 5 — Removing Missing Values

**Signature:** `DataFrame.dropna()`

Instead of filling missing values, you can remove them entirely.

### 5.1 Remove Rows with Missing Values

`dropna()` by default removes **any row** that contains at least one NaN.

In [ ]:
# Remove any row with a missing value
clean = df.dropna()
print("Original shape:", df.shape)
print("After dropna() shape:", clean.shape)
print("\nClean DataFrame:")
print(clean)

**⚠️ Warning:** `dropna()` can delete a lot of data. Use it carefully.

### 5.2 Remove Columns with Missing Values

Use `axis=1` to drop columns instead of rows.

In [ ]:
# Remove columns that have ANY missing value
clean_cols = df.dropna(axis=1)
print("Original columns:", df.columns.tolist())
print("After dropna(axis=1):", clean_cols.columns.tolist())
print("\nClean DataFrame:")
print(clean_cols)

**Remember:**
- `axis=0` — rows (default)
- `axis=1` — columns

### 5.3 Remove Only if ALL Values Missing

`how="all"` removes rows where **every** value is NaN.

In [ ]:
# Create a row with all NaN
df_all_nan = df.copy()
df_all_nan.loc[5] = np.nan  # Add a completely empty row
print("With empty row:")
print(df_all_nan)

# Remove only rows where ALL values are NaN
print("\nAfter dropna(how='all'):")
print(df_all_nan.dropna(how="all"))

### 5.4 Remove if Specific Column Missing

Use `subset` to only check certain columns.

In [ ]:
# Remove rows only if Age is missing
subset_drop = df.dropna(subset=["Age"])
print("Original:")
print(df)
print("\nAfter dropna(subset=['Age']):")
print(subset_drop)

In [ ]:
# Remove rows if Age OR Python is missing
subset_drop2 = df.dropna(subset=["Age", "Python"])
print("After dropna(subset=['Age', 'Python']):")
print(subset_drop2)

### 5.5 Threshold — Minimum Number of Non-NaN Values

`thresh=n` keeps rows with at least `n` non-NaN values.

In [ ]:
# Keep rows with at least 4 non-NaN values
thresh_drop = df.dropna(thresh=4)
print("After dropna(thresh=4) — needs at least 4 non-NaN:")
print(thresh_drop)

---
## PART 6 — Duplicates

Our dataset has Alice appearing twice with identical values.

**Problems duplicates cause:**
- Skew statistics (mean, sum, etc.)
- Cause data leakage in ML (same data in train and test)
- Waste memory and storage

### 6.1 Detect Duplicates

**Signature:** `DataFrame.duplicated()`

Returns a boolean Series — `True` for rows that are **duplicates** of previous rows.

In [ ]:
# Detect duplicate rows
print("Duplicate mask:")
print(df.duplicated())

Notice that the first Alice (row 0) is **not** flagged — only the second Alice (row 4) is marked as duplicate.

### 6.2 Count Duplicates

In [ ]:
# Count duplicates
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

### 6.3 View Duplicate Rows

In [ ]:
# Show the duplicated rows
print("Duplicate rows:")
print(df[df.duplicated()])

### 6.4 Remove Duplicates

**Signature:** `DataFrame.drop_duplicates()`

In [ ]:
# Remove duplicate rows
df_no_dupes = df.drop_duplicates()
print("Original shape:", df.shape)
print("After drop_duplicates() shape:", df_no_dupes.shape)
print("\nClean DataFrame:")
print(df_no_dupes)

### 6.5 Remove Based on a Single Column

Use `subset` to only check one column for duplicates.

In [ ]:
# Remove duplicates based on Name only
df_name_dedup = df.drop_duplicates(subset=["Name"])
print("Drop duplicates based on Name:")
print(df_name_dedup)

### 6.6 Keep First or Last

Use `keep` parameter:
- `keep="first"` — keep the first occurrence (default)
- `keep="last"` — keep the last occurrence
- `keep=False` — drop **all** duplicates (keep none)

In [ ]:
# Keep last occurrence instead of first
df_last = df.drop_duplicates(keep="last")
print("Keep last duplicate:")
print(df_last)

In [ ]:
# Drop ALL rows that have any duplicate (remove both)
df_none = df.drop_duplicates(keep=False)
print("Drop all duplicates (keep=False):")
print(df_none)

---
## PART 7 — Replacing Values

**Signature:** `DataFrame.replace(to_replace, value)`

Used when values need to be corrected or standardized.

### 7.1 Replace a Single Value

In [ ]:
# Replace "CS" with "Computer Science" in Department
df_replace = df.copy()
df_replace["Department"] = df_replace["Department"].replace("CS", "Computer Science")
df_replace["Department"] = df_replace["Department"].replace("SE", "Software Engineering")
print(df_replace)

### 7.2 Replace Multiple Values with a Dictionary

In [ ]:
# Replace multiple values at once
df_replace_multi = df.copy()
df_replace_multi["Department"] = df_replace_multi["Department"].replace({
    "CS": "Computer Science",
    "IT": "Information Technology",
    "SE": "Software Engineering"
})
print(df_replace_multi)

### 7.3 Replace Across the Entire DataFrame

In [ ]:
# Replace in ALL columns at once
df_global_replace = df.copy()
df_global_replace = df_global_replace.replace({
    "CS": "Computer Science",
    "IT": "Information Technology",
    "SE": "Software Engineering"
})
print(df_global_replace)

### 7.4 Replace with Regular Expressions

Use `regex=True` to use pattern matching.

In [ ]:
# Replace anything starting with 'C' with 'Department_C'
df_regex = df.copy()
df_regex["Department"] = df_regex["Department"].replace("^C.*", "Department_C", regex=True)
print(df_regex)

---
## PART 8 — Changing Data Types

**Signature:** `Series.astype(dtype)`

Sometimes Pandas assigns the wrong type to a column.
For example, Age might become `float64` because of NaN values.

In [ ]:
# Check current dtypes
print("Current data types:")
print(df.dtypes)

### 8.1 Convert to Integer

**Note:** You cannot convert a column with NaN to int directly.
Fill NaN first, then convert.

In [ ]:
# Fill NaN first, then convert to int
df_int = df.copy()
df_int["Age"] = df_int["Age"].fillna(df_int["Age"].mean()).astype(int)
print("Age as int:")
print(df_int["Age"])
print(f"dtype: {df_int['Age'].dtype}")

### 8.2 Convert to Float

In [ ]:
# Convert to float
df_float = df.copy()
df_float["Python"] = df_float["Python"].astype(float)
print("Python as float:")
print(df_float["Python"])
print(f"dtype: {df_float['Python'].dtype}")

### 8.3 Convert to String

In [ ]:
# Convert Age to string
df_str = df.copy()
df_str["Age"] = df_str["Age"].astype(str)
print("Age as string:")
print(df_str["Age"])
print(f"dtype: {df_str['Age'].dtype}")

### 8.4 Convert to Boolean

In [ ]:
# Create a boolean column
df_bool = df.copy()
df_bool["Passed"] = df_bool["Python"] > 80
print("Boolean column (Python > 80):")
print(df_bool[["Name", "Python", "Passed"]])
print(f"dtype: {df_bool['Passed'].dtype}")

### 8.5 Category Type

The `category` dtype saves memory when you have repeated text values.

In [ ]:
# Convert Department to category
df_cat = df.copy()
df_cat["Department"] = df_cat["Department"].astype("category")
print("Department as category:")
print(df_cat["Department"])
print(f"dtype: {df_cat['Department'].dtype}")

In [ ]:
# Compare memory usage
print("Memory usage (object):", df["Department"].memory_usage(deep=True), "bytes")
print("Memory usage (category):", df_cat["Department"].memory_usage(deep=True), "bytes")

### 8.6 Common Conversions Reference

```python
astype(int)       # Convert to integer
astype(float)     # Convert to float
astype(str)       # Convert to string
astype(bool)      # Convert to boolean
astype("category") # Convert to category (saves memory)
```

---
## PART 9 — Renaming Columns

**Signature:** `DataFrame.rename(columns={old: new})`

Renaming columns makes your dataset more readable.

In [ ]:
# Rename a single column
df_rename = df.copy()
df_rename = df_rename.rename(columns={"AI": "Artificial_Intelligence"})
print(df_rename.columns.tolist())
print(df_rename)

In [ ]:
# Rename multiple columns at once
df_rename_multi = df.copy()
df_rename_multi = df_rename_multi.rename(columns={
    "Age": "Student_Age",
    "Python": "Python_Score",
    "AI": "AI_Score"
})
print(df_rename_multi.columns.tolist())
print(df_rename_multi)

### Alternative: Set columns directly

In [ ]:
# Alternative: directly assign new column names
df_cols = df.copy()
df_cols.columns = ["Student", "Years", "PyScore", "AIScore", "Dept"]
print(df_cols)

### Rename with a Function

Use `str.upper()`, `str.lower()`, or `str.strip()` on column names.

In [ ]:
# Convert all column names to uppercase
df_upper = df.copy()
df_upper.columns = [col.upper() for col in df_upper.columns]
print(df_upper.columns.tolist())
print(df_upper)

---
## PART 10 — String Operations

One of Pandas' best features.

All string methods begin with `.str`.

**Important:** These only work on columns with dtype `object` (string).

In [ ]:
# Create a dataset with messy strings
messy_names = pd.DataFrame({
    "Name": ["  Alice  ", "BOB", "charlie", "  DAVID  ", "eVe"]
})
print("Messy names:")
print(messy_names)

### 10.1 Lowercase

`.str.lower()` converts all characters to lowercase.

In [ ]:
messy_names["Name_Lower"] = messy_names["Name"].str.lower()
print(messy_names)

### 10.2 Uppercase

`.str.upper()` converts all characters to uppercase.

In [ ]:
messy_names["Name_Upper"] = messy_names["Name"].str.upper()
print(messy_names)

### 10.3 Title Case

`.str.title()` capitalizes the first letter of each word.

In [ ]:
messy_names["Name_Title"] = messy_names["Name"].str.title()
print(messy_names)

### 10.4 Length

`.str.len()` returns the length of each string (number of characters).

In [ ]:
messy_names["Name_Length"] = messy_names["Name"].str.len()
print(messy_names)

### 10.5 Contains

`.str.contains(pattern)` checks if a string contains a substring.

In [ ]:
messy_names["Has_A"] = messy_names["Name"].str.contains("a", case=False)
print(messy_names)

### 10.6 Starts With / Ends With

In [ ]:
# Check if string starts with "A" (case sensitive)
messy_names["Starts_A"] = messy_names["Name"].str.startswith("A")
print(messy_names)

In [ ]:
# Check if string ends with "e"
messy_names["Ends_E"] = messy_names["Name"].str.endswith("e")
print(messy_names)

### 10.7 Replace Text

`.str.replace(old, new)` replaces substrings within strings.

In [ ]:
# Replace 'e' with 'E' in strings
messy_names["Name_Replaced"] = messy_names["Name"].str.replace("e", "E")
print(messy_names)

### 10.8 Remove Spaces (Strip)

`.str.strip()` removes leading and trailing whitespace.

This is **extremely common** in real-world data cleaning.

In [ ]:
# Strip whitespace
messy_names["Name_Stripped"] = messy_names["Name"].str.strip()
print("After strip:")
print(messy_names[["Name", "Name_Stripped"]])

In [ ]:
# Strip + title case combined (common cleaning chain)
messy_names["Name_Clean"] = messy_names["Name"].str.strip().str.title()
print("Fully cleaned names:")
print(messy_names[["Name", "Name_Clean"]])

### 10.9 Split Strings

`.str.split(separator)` splits strings into lists.

In [ ]:
# Split full names
names_df = pd.DataFrame({
    "Full_Name": ["John Doe", "Mary Jane", "Peter Pan"]
})
names_df["Name_Split"] = names_df["Full_Name"].str.split(" ")
print(names_df)

In [ ]:
# Split into separate columns using expand=True
names_df[["First", "Last"]] = names_df["Full_Name"].str.split(" ", expand=True)
print(names_df)

### 10.10 Extract Substrings

`.str.extract(pattern)` extracts matching groups using regex.

In [ ]:
# Extract first letter from each name
names_df["First_Letter"] = names_df["First"].str.extract(r"(^.)")
print(names_df)

### 10.11 Check if Strings are Numeric

`.str.isnumeric()` checks if all characters are digits.

In [ ]:
mixed = pd.Series(["123", "abc", "456", "12a", "   ", ""])
print("Original:", mixed.tolist())
print("Is numeric:", mixed.str.isnumeric().tolist())

---
## PART 11 — Unique Values

Understanding what values exist in your data is essential for cleaning.

In [ ]:
# Our working dataset (after removing duplicates)
df_clean = df.drop_duplicates()
print(df_clean)

### 11.1 unique()

`unique()` returns an array of all **distinct values** in a column.

In [ ]:
# Get unique departments
unique_depts = df_clean["Department"].unique()
print("Unique departments:", unique_depts)
print(f"Type: {type(unique_depts)}")

### 11.2 nunique()

`nunique()` returns the **count** of distinct values.

In [ ]:
# Count unique departments
num_unique = df_clean["Department"].nunique()
print(f"Number of unique departments: {num_unique}")

### 11.3 value_counts()

`value_counts()` returns the **frequency** of each unique value.

This is **one of the most useful methods** for exploratory data analysis.

In [ ]:
# Count occurrences of each department
dept_counts = df_clean["Department"].value_counts()
print("Department value counts:")
print(dept_counts)

In [ ]:
# value_counts with normalize=True gives proportions
dept_props = df_clean["Department"].value_counts(normalize=True)
print("Department proportions:")
print(dept_props)

In [ ]:
# value_counts also works on numeric columns
print("Age value counts:")
print(df_clean["Age"].value_counts().sort_index())

### Key Differences

| Method | Returns | Use Case |
|--------|---------|----------|
| `unique()` | Array of values | "What values exist?" |
| `nunique()` | Single integer | "How many distinct values?" |
| `value_counts()` | Series (values → counts) | "How frequent is each value?" |

---
## PART 12 — Clip Outliers

**Signature:** `Series.clip(lower=None, upper=None)`

Sometimes values are unrealistic (outliers).
Clipping caps them at a minimum/maximum value.

In [ ]:
# Create salary data with an unrealistic outlier
salaries = pd.DataFrame({
    "Employee": ["John", "Mary", "Peter", "Jane", "Mike"],
    "Salary": [5000, 6000, 5500, 9000000, 5800]  # 9,000,000 is an outlier
})
print("Original salaries:")
print(salaries)

In [ ]:
# Clip the outlier — cap at upper limit
salaries["Salary_Clipped"] = salaries["Salary"].clip(upper=10000)
print("After clipping (upper=10000):")
print(salaries)

In [ ]:
# Clip with both lower and upper bounds
salaries["Salary_Bounded"] = salaries["Salary"].clip(lower=4000, upper=8000)
print("After clipping (lower=4000, upper=8000):")
print(salaries)

In [ ]:
# Practical: clip Age to a reasonable range
df_clip = df.copy()
df_clip.loc[5] = ["Eve", 150, 95, 88, "CS"]  # Age 150 is unrealistic
print("With outlier:")
print(df_clip)

df_clip["Age"] = df_clip["Age"].clip(upper=40)
print("\nAfter clip(upper=40):")
print(df_clip)

---
## PART 13 — Common Cleaning Workflow

This is a realistic workflow you'll see in many real-world projects.

In [ ]:
# Simulate loading a messy CSV
messy_data = {
    "Name": ["  John  ", "Mary", "PETER", "  John  ", "Anna", np.nan],
    "Age": [25, np.nan, 22, 25, 30, 28],
    "Department": ["IT", "HR", "IT", "IT", None, "Finance"],
    "Salary": [5000, 4000, np.nan, 5000, 6000, 7000]
}
df_raw = pd.DataFrame(messy_data)

print("RAW DATASET:")
print(df_raw)
print("\n" + "="*60)
print("STARTING CLEANING WORKFLOW...")
print("="*60)

In [ ]:
# Step 1: Inspect the data
print("--- Step 1: Inspect ---")
print(df_raw.info())
print()
print(df_raw.describe())

In [ ]:
# Step 2: Check missing values
print("--- Step 2: Missing Values ---")
print(df_raw.isna().sum())

In [ ]:
# Step 3: Remove duplicates
print("--- Step 3: Remove Duplicates ---")
print(f"Duplicates before: {df_raw.duplicated().sum()}")
df_work = df_raw.drop_duplicates()
print(f"Shape after: {df_work.shape}")

In [ ]:
# Step 4: Fill missing values
print("--- Step 4: Fill Missing Values ---")
df_work["Age"] = df_work["Age"].fillna(df_work["Age"].mean())
df_work["Salary"] = df_work["Salary"].fillna(df_work["Salary"].median())
df_work["Department"] = df_work["Department"].fillna("Unknown")
df_work["Name"] = df_work["Name"].fillna("Unknown")
print("Missing values filled.")
print(df_work.isna().sum())

In [ ]:
# Step 5: Clean strings
print("--- Step 5: Clean Strings ---")
df_work["Name"] = df_work["Name"].str.strip().str.title()
print("Names cleaned.")

In [ ]:
# Step 6: Convert types
print("--- Step 6: Type Conversion ---")
df_work["Age"] = df_work["Age"].astype(int)
print(f"Age dtype: {df_work['Age'].dtype}")

In [ ]:
# Step 7: Rename columns
print("--- Step 7: Rename Columns ---")
df_work = df_work.rename(columns={"Salary": "Monthly_Salary"})
print(f"Columns: {df_work.columns.tolist()}")

In [ ]:
# Final: Display cleaned dataset
print("--- FINAL CLEANED DATASET ---")
print(df_work)
print("\n--- Summary ---")
print(f"Shape: {df_work.shape}")
print(f"Missing values: {df_work.isna().sum().sum()}")
print(f"Dtypes:\n{df_work.dtypes}")

---
## Day 3 Cheat Sheet

```python
# MISSING VALUES
df.isna()                  # Detect NaN
df.isnull()                # Same as isna()
df.isna().sum()            # Count per column
df.isna().sum().sum()      # Total count

# FILLING
df.fillna(0)               # Fill with constant
df["col"] = df["col"].fillna(value)  # Fill one column
df["col"] = df["col"].fillna(df["col"].mean())  # Fill with mean
df["col"] = df["col"].fillna(df["col"].median())  # Fill with median
df.ffill()                 # Forward fill
df.bfill()                 # Backward fill

# REMOVING
df.dropna()                # Drop rows with NaN
df.dropna(axis=1)          # Drop columns with NaN
df.dropna(subset=["col"])  # Drop if specific column NaN
df.dropna(thresh=n)        # Keep if >= n non-NaN values

# DUPLICATES
df.duplicated()            # Detect duplicates
df.duplicated().sum()      # Count duplicates
df.drop_duplicates()       # Remove duplicates
df.drop_duplicates(subset=["col"])  # Remove by column
df.drop_duplicates(keep="last")     # Keep last occurrence

# REPLACE
df.replace(old, new)       # Replace single value
df.replace({old: new})     # Replace with dict
df["col"].replace(regex)  # Regex replace

# TYPE CONVERSION
df["col"].astype(int)      # To integer
df["col"].astype(float)    # To float
df["col"].astype(str)      # To string
df["col"].astype("category")  # To category

# RENAME
df.rename(columns={"old": "new"})  # Rename columns

# STRING METHODS
df["col"].str.lower()      # Lowercase
df["col"].str.upper()      # Uppercase
df["col"].str.title()      # Title case
df["col"].str.strip()      # Remove spaces
df["col"].str.contains("x")  # Check substring
df["col"].str.startswith("x")  # Starts with
df["col"].str.endswith("x")    # Ends with
df["col"].str.replace(a, b)    # Replace substring
df["col"].str.len()        # String length
df["col"].str.split(" ")   # Split string

# UNIQUE & COUNTS
df["col"].unique()         # Array of unique values
df["col"].nunique()        # Count of unique values
df["col"].value_counts()   # Frequency of each value

# OUTLIERS
df["col"].clip(lower=x, upper=y)  # Clip values
```

---
## Day 3 Knowledge Check

Answer these questions in your own words:

**1.** What is the difference between `isna()` and `isnull()`?

**2.** What does `fillna()` do?

**3.** What is the difference between `dropna()` and `fillna()`?

**4.** What does `duplicated()` return?

**5.** What is the difference between `unique()` and `value_counts()`?

**6.** Why do we use `astype()`?

**7.** What does `.str.strip()` do?

**8.** When would you use `dropna(axis=1)`?

**9.** What is the difference between `keep="first"` and `keep=False` in `drop_duplicates()`?

**10.** Why can't you convert a column with NaN directly to `int`?

**Your answers:**

**1.** There is no difference — `isna()` and `isnull()` are identical aliases.

**2.** `fillna()` replaces missing (NaN) values with a specified value.

**3.** `dropna()` removes rows/columns with NaN; `fillna()` keeps them but fills the NaN with a value.

**4.** `duplicated()` returns a boolean Series where `True` means the row is a duplicate of a previous row.

**5.** `unique()` returns an array of distinct values; `value_counts()` returns a Series of distinct values with their frequencies.

**6.** `astype()` converts a column from one data type to another (e.g., float to int).

**7.** `.str.strip()` removes leading and trailing whitespace from strings.

**8.** When you want to remove entire columns that contain missing values instead of removing rows.

**9.** `keep="first"` keeps the first occurrence and removes later duplicates; `keep=False` removes ALL rows that have duplicates (both the first and later ones).

**10.** NaN is a float value, and integers in Pandas cannot represent NaN. You must fill NaN first, then convert to int.

---
## 💻 Coding Challenge

Use the dataset below and complete the tasks.

In [ ]:
import pandas as pd
import numpy as np

employees = {
    "Name": ["John", "Mary", "Peter", "John", " Anna "],
    "Age": [25, np.nan, 22, 25, 28],
    "Department": ["IT", "HR", "IT", "IT", None],
    "Salary": [5000, 4000, np.nan, 5000, 6000]
}

df = pd.DataFrame(employees)
print("EMPLOYEE DATASET:")
print(df)

EMPLOYEE DATASET:
     Name   Age Department  Salary
0    John  25.0         IT  5000.0
1    Mary   NaN         HR  4000.0
2   Peter  22.0         IT     NaN
3    John  25.0         IT  5000.0
4   Anna   28.0       None  6000.0


: 

**Tasks:**
1. Display missing values (`isna()`)
2. Count missing values per column
3. Fill missing Age with the mean
4. Fill missing Salary with the median
5. Fill missing Department with "Unknown"
6. Remove duplicate rows
7. Remove spaces from Name
8. Convert Age to integer
9. Rename Salary → Monthly_Salary
10. Display unique departments
11. Count employees per department

In [ ]:
# Your code here:

# 1. Display missing values
print("Missing values:")
print(df.isna())
print()

# 2. Count missing values per column
print("Missing values count:")
print(df.isna().sum())
print()

# 3. Fill missing Age with mean
df["Age"] = df["Age"].fillna(df["Age"].mean())

# 4. Fill missing Salary with median
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

# 5. Fill missing Department with "Unknown"
df["Department"] = df["Department"].fillna("Unknown")

# 6. Remove duplicate rows
df = df.drop_duplicates()

# 7. Remove spaces from Name
df["Name"] = df["Name"].str.strip()

# 8. Convert Age to integer
df["Age"] = df["Age"].astype(int)

# 9. Rename Salary -> Monthly_Salary
df = df.rename(columns={"Salary": "Monthly_Salary"})

# 10. Display unique departments
print("Unique departments:", df["Department"].unique())
print()

# 11. Count employees per department
print("Employees per department:")
print(df["Department"].value_counts())
print()

# Final display
print("CLEANED DATASET:")
print(df)

Missing values:
    Name    Age  Department  Salary
0  False  False       False   False
1  False   True       False   False
2  False  False       False    True
3  False  False       False   False
4  False  False        True   False

Missing values count:
Name          0
Age           1
Department    1
Salary        1
dtype: int64

Unique departments: ['IT' 'HR' 'Unknown']

Employees per department:
Department
IT         2
HR         1
Unknown    1
Name: count, dtype: int64

CLEANED DATASET:
    Name  Age Department  Monthly_Salary
0   John   25         IT          5000.0
1   Mary   25         HR          4000.0
2  Peter   22         IT          5000.0
4   Anna   28    Unknown          6000.0


---
## 🚀 Mini Project — Employee Data Cleaner

Apply everything you've learned in a complete workflow.

In [1]:
import pandas as pd
import numpy as np

employees = {
    "Name": [
        "John ",
        " Mary",
        "Peter",
        "John ",
        "Anna",
        "Mike "
    ],
    "Age": [25, np.nan, 22, 25, 30, np.nan],
    "Department": [
        "IT",
        "HR",
        "IT",
        "IT",
        None,
        "Finance"
    ],
    "Salary": [
        5000,
        4000,
        np.nan,
        5000,
        6000,
        7000
    ]
}

df = pd.DataFrame(employees)
print("RAW DATASET:")
print(df)

RAW DATASET:
    Name   Age Department  Salary
0  John   25.0         IT  5000.0
1   Mary   NaN         HR  4000.0
2  Peter  22.0         IT     NaN
3  John   25.0         IT  5000.0
4   Anna  30.0       None  6000.0
5  Mike    NaN    Finance  7000.0


**Your program should:**
1. Display `info()`
2. Display missing values
3. Count missing values
4. Fill missing Age using the mean
5. Fill missing Salary using the median
6. Fill missing Department with "Unknown"
7. Remove duplicate rows
8. Remove spaces from Name
9. Rename Salary to Monthly_Salary
10. Convert Age to integer
11. Display `value_counts()` for Department
12. Save as `clean_employees.csv`
13. Load the CSV again
14. Display the cleaned DataFrame

In [2]:
# Your mini project code here:

# 1. Display info
print("=" * 60)
print("STEP 1: Dataset Info")
print("=" * 60)
print(df.info())
print()

# 2. Display missing values
print("=" * 60)
print("STEP 2: Missing Values")
print("=" * 60)
print(df.isna())
print()

# 3. Count missing values
print("=" * 60)
print("STEP 3: Missing Values Count")
print("=" * 60)
print(df.isna().sum())
print()

# 4. Fill missing Age using the mean
print("=" * 60)
print("STEP 4: Fill Age with Mean")
print("=" * 60)
mean_age = df["Age"].mean()
print(f"Mean Age: {mean_age:.2f}")
df["Age"] = df["Age"].fillna(mean_age)
print()

# 5. Fill missing Salary using the median
print("=" * 60)
print("STEP 5: Fill Salary with Median")
print("=" * 60)
median_salary = df["Salary"].median()
print(f"Median Salary: {median_salary}")
df["Salary"] = df["Salary"].fillna(median_salary)
print()

# 6. Fill missing Department with "Unknown"
print("=" * 60)
print("STEP 6: Fill Department with Unknown")
print("=" * 60)
df["Department"] = df["Department"].fillna("Unknown")
print("Done.")
print()

# 7. Remove duplicate rows
print("=" * 60)
print("STEP 7: Remove Duplicates")
print("=" * 60)
print(f"Duplicates found: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after: {df.shape}")
print()

# 8. Remove spaces from Name
print("=" * 60)
print("STEP 8: Clean Names")
print("=" * 60)
df["Name"] = df["Name"].str.strip().str.title()
print("Names cleaned.")
print()

# 9. Rename Salary to Monthly_Salary
print("=" * 60)
print("STEP 9: Rename Column")
print("=" * 60)
df = df.rename(columns={"Salary": "Monthly_Salary"})
print(f"Columns: {df.columns.tolist()}")
print()

# 10. Convert Age to integer
print("=" * 60)
print("STEP 10: Convert Age to int")
print("=" * 60)
df["Age"] = df["Age"].astype(int)
print(f"Age dtype: {df['Age'].dtype}")
print()

# 11. Display value_counts for Department
print("=" * 60)
print("STEP 11: Department Value Counts")
print("=" * 60)
print(df["Department"].value_counts())
print()

# 12. Save as clean_employees.csv
print("=" * 60)
print("STEP 12: Save to CSV")
print("=" * 60)
df.to_csv("clean_employees.csv", index=False)
print("Saved as clean_employees.csv")
print()

# 13. Load the CSV again
print("=" * 60)
print("STEP 13: Load CSV")
print("=" * 60)
df_loaded = pd.read_csv("clean_employees.csv")
print("Loaded successfully.")
print()

# 14. Display the cleaned DataFrame
print("=" * 60)
print("STEP 14: FINAL CLEANED DATASET")
print("=" * 60)
print(df_loaded)
print()
print("✅ Mini project complete!")

STEP 1: Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Name        6 non-null      object 
 1   Age         4 non-null      float64
 2   Department  5 non-null      object 
 3   Salary      5 non-null      float64
dtypes: float64(2), object(2)
memory usage: 324.0+ bytes
None

STEP 2: Missing Values
    Name    Age  Department  Salary
0  False  False       False   False
1  False   True       False   False
2  False  False       False    True
3  False  False       False   False
4  False  False        True   False
5  False   True       False   False

STEP 3: Missing Values Count
Name          0
Age           2
Department    1
Salary        1
dtype: int64

STEP 4: Fill Age with Mean
Mean Age: 25.50

STEP 5: Fill Salary with Median
Median Salary: 5000.0

STEP 6: Fill Department with Unknown
Done.

STEP 7: Remove Duplicates
Duplicates found: 

---
## Day 3 Summary

Today you learned the essential data-cleaning tools used before analysis or Machine Learning:

| Category | Methods |
|----------|---------|
| Detect missing | `isna()`, `isnull()`, `notna()` |
| Fill missing | `fillna()`, `ffill()`, `bfill()` |
| Remove missing | `dropna()` |
| Duplicates | `duplicated()`, `drop_duplicates()` |
| Replace | `replace()` |
| Type conversion | `astype()` |
| Rename | `rename()` |
| String methods | `.str.lower()`, `.str.strip()`, `.str.contains()`, etc. |
| Unique values | `unique()`, `nunique()`, `value_counts()` |
| Outliers | `clip()` |

These are among the most frequently used Pandas methods in real-world projects.

By mastering them, you can take messy datasets and turn them into clean, analysis-ready data.

🚀 **Next: Day 4 — Data Transformation & Grouping**